In [66]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition
from analysis import fit_peak_expbg

In [67]:
# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = 1.460
TL208_MEV = 2.614
CS137_MEV = 0.661
colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [68]:
files1 = sorted((project_root/'data/260317').glob('1*.csv'))
acqs1 = [SiphraAcquisition(file) for file in files1]

In [69]:
acqs1.pop(3)
acqs1.pop(3)
acqs1

[SIPHRAAcquisition(File: '10_GainTesting_SiPM_Vbias29,0V_All8Chs_Na22_SIPHRA1'),
 SIPHRAAcquisition(File: '11_GainTesting_SiPM_Vbias29,0V_All8Chs_Na22_SIPHRA1'),
 SIPHRAAcquisition(File: '12_GainTesting_SiPM_Vbias29,0V_All8Chs_Na22_SIPHRA1'),
 SIPHRAAcquisition(File: '15_GainTesting_SiPM_Vbias29,0V_All8Chs_Na22_SIPHRA1'),
 SIPHRAAcquisition(File: '16_GainTesting_SiPM_Vbias29,0V_All8Chs_Background_SIPHRA1'),
 SIPHRAAcquisition(File: '17_GainTesting_SiPM_Vbias29,0V_All8Chs_Background_SIPHRA1'),
 SIPHRAAcquisition(File: '18_GainTesting_SiPM_Vbias29,0V_All8Chs_Background_SIPHRA1')]

In [70]:
acqs1[0].filepath.stem[:2]

'10'

In [71]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lg = ROOT.TLegend(0.58, 0.61, 0.75, 0.83)
lg.SetHeader('CMIS gain')

ci_gain='1V/0.75nC'
cmis_gains=['1/10', '1/100', '1/200', '1/400']
hists={}

for idx, (acq, cmsg) in enumerate(zip(acqs1[:4], cmis_gains)):
    hist = ROOT.TH1F(f"cmis_gain {cmsg}", f"{cmsg}", BITS12, 0, BITS12)
    hist.Fill(acq['s']/len(acq.active_chs))
    hist.Scale(1/acq.exposure)
    # Preeting up ...
    hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
    hist.GetXaxis().SetTitle("ADC channel number")
    hist.GetYaxis().SetTitle(r"Count rate (s^{-1})")
    hist.SetLineColor(colors[idx]+2)
    # legend and storing
    lg.AddEntry(hist, f"{cmsg}")
    hists[f"Acq{acq.filepath.stem[:2]}"] = hist
    del hist
    hists[f"Acq{acq.filepath.stem[:2]}"].Draw('sames hist')
cv.SetLogy()
lg.Draw()
cv.Draw()
ROOT.gPad.Modified()
ROOT.gPad.Update()

Warning in <TROOT::Append>: Replacing existing TH1: cmis_gain 1/10 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: cmis_gain 1/100 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: cmis_gain 1/200 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: cmis_gain 1/400 (Potential memory leak).


In [72]:
# Calibration with acquisition 15(Signal) and 16 (BG)
files2 = sorted([p for p in (project_root/'data/260317').iterdir() if p.stem[11:13] in ['15', '16']])
acqs2 = [SiphraAcquisition(file) for file in files2]

In [73]:
files2

[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260317/SUBTRACTED_15_GainTesting_SiPM_Vbias29,0V_All8Chs_Na22_SIPHRA1.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260317/SUBTRACTED_16_GainTesting_SiPM_Vbias29,0V_All8Chs_Background_SIPHRA1.csv')]

In [74]:
len(acqs2[0].active_chs)

8

In [76]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lg = ROOT.TLegend(0.58, 0.61, 0.75, 0.83)
# lg.SetHeader('')

ci_gain='1V/0.75nC'
labels=['Signal', 'Background']

ref_exposure = acqs2[0].exposure

for idx, (acq, label) in enumerate(zip(acqs2, labels)):
    hist = ROOT.TH1F(label, label, BITS12, 0, BITS12)
    hist.Fill(acq['s']/len(acq.active_chs))
    # hist.Scale(ref_exposure/acq.exposure)
    # Preeting up ...
    hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC, CMIS gain = 1/400")
    hist.GetXaxis().SetTitle("ADC channel number")
    hist.GetYaxis().SetTitle(r"Counts")
    hist.SetLineColor(colors[idx]+2)
    # legend and storing
    lg.AddEntry(hist, label)
    hists[f"Acq{acq.filepath.stem[:13]}"] = hist
    del hist
    hists[f"Acq{acq.filepath.stem[:13]}"].Draw('sames hist')

# hist_subt = hists['AcqSUBTRACTED_15'].Clone('Subtracted')
# hist_subt.Add(hists['AcqSUBTRACTED_16'], -1)
# hist_subt.Draw('sames hist')
# hist_subt.SetLineColor(colors[2])
# lg.AddEntry(hist_subt, 'Subtracted')

cv.SetLogy()
lg.Draw()
cv.Draw()
ROOT.gPad.Modified()
ROOT.gPad.Update()
print(f"Exposures:\n    Signal:\t{acqs2[0].exposure:>.3f} s\n    Background:\t{acqs2[1].exposure:>.3f} s")

Exposures:
    Signal:	220.216 s
    Background:	625.392 s


Warning in <TROOT::Append>: Replacing existing TH1: Signal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Background (Potential memory leak).


In [50]:
hists

{'Acq10': <cppyy.gbl.TH1F object at 0x598c183055b0>,
 'Acq11': <cppyy.gbl.TH1F object at 0x598c182bf720>,
 'Acq12': <cppyy.gbl.TH1F object at 0x598c182bfb10>,
 'Acq15': <cppyy.gbl.TH1F object at 0x598c182ef160>,
 'AcqSUBTRACTED_10': <cppyy.gbl.TH1F object at 0x598c18212f00>,
 'AcqSUBTRACTED_11': <cppyy.gbl.TH1F object at 0x598c19acd960>,
 'AcqSUBTRACTED_16': <cppyy.gbl.TH1F object at 0x598c18bf0f40>,
 'AcqSUBTRACTED_15': <cppyy.gbl.TH1F object at 0x598c18108c10>}

In [82]:
files2 = sorted((project_root/'data/260317').glob('SUBTRACTED_*'))
acqs2 = [SiphraAcquisition(file) for file in files2]
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lg = ROOT.TLegend(0.58, 0.61, 0.75, 0.83)

hist = hist_subt.Clone('hist_subt_calib')
hist.SetTitle(r'^{22}Na subtracted spectrum, CI gain = 1/0.75 pC, CMIS gain = 1/400')
fit1 = hist.Fit('gaus', 'L S', "", 100, 300)
hist.Draw()
cv.SetLogy()
cv.Draw()

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      7605.91
Chi2                      =      15211.8
NDf                       =          197
Edm                       =  1.10191e-06
NCalls                    =          138
Constant                  =      1267.88   +/-   5.7826      
Mean                      =      172.076   +/-   0.0854468   
Sigma                     =      22.9615   +/-   0.0616869    	 (limited)


In [ ]:
# Calibration with acquisition 15(Signal) and 16 (BG)
files2 = sorted((project_root/'data/260317').glob('SUBTRACTED_*'))
acqs2 = [SiphraAcquisition(file) for file in files2]
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lg = ROOT.TLegend(0.58, 0.61, 0.75, 0.83)
# lg.SetHeader('CMIS gain')

ci_gain='1V/0.75nC'
labels=['Signal', 'Background']
hists={}

for idx, (acq, label) in enumerate(zip(acqs2, labels)):
    hist = ROOT.TH1F(label, label, BITS12, 0, BITS12)
    hist.Fill(acq['s']/len(acq.active_chs))
    hist.Scale(1/acq.exposure)
    # Preeting up ...
    hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC, CMIS gain = 1/400")
    hist.GetXaxis().SetTitle("ADC channel number")
    hist.GetYaxis().SetTitle(r"Count rate (s^{-1})")
    hist.SetLineColor(colors[idx]+2)
    # legend and storing
    lg.AddEntry(hist, label)
    hists[f"Acq{acq.filepath.stem[:13]}"] = hist
    del hist
    hists[f"Acq{acq.filepath.stem[:13]}"].Draw('sames hist')



cv.SetLogy()
lg.Draw()
cv.Draw()
ROOT.gPad.Modified()
ROOT.gPad.Update()

In [56]:
2278 - 274

2004

In [52]:
1841 - 637

1204

In [55]:
984-424

560